# Part 1 - human groundedness evaluation (both members)

In [ ]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

In [ ]:
# Part 1 human evaluation: groundedness 1-5 by both members on the 25 final (hybrid) answers
import pandas as pd
from pathlib import Path
ROOT = Path.cwd()
for cand in [ROOT, ROOT.parent, Path("/app/DATA266-Lab2-Pair45")]:
    if (cand / "part1_rag").exists(): ROOT = cand; break
OUT = ROOT / "part1_rag" / "Nikhil_Kanaparthi" / "outputs"
a = pd.read_csv(OUT / "human_groundedness_nikhil.csv"); b = pd.read_csv(OUT / "human_groundedness_samruddhi.csv")
m = a[["question_id", "rating"]].merge(b[["question_id", "rating"]], on="question_id", suffixes=("_nikhil", "_samruddhi"))
m["diff"] = (m.rating_nikhil - m.rating_samruddhi).abs()
res = {"average_groundedness": round(float(m[["rating_nikhil", "rating_samruddhi"]].stack().mean()), 2),
       "average_nikhil": round(float(m.rating_nikhil.mean()), 2), "average_samruddhi": round(float(m.rating_samruddhi.mean()), 2),
       "exact_agreement": round(float((m["diff"] == 0).mean()), 3), "agreement_within_1": round(float((m["diff"] <= 1).mean()), 3), "n": len(m)}
pd.Series(res).to_json(OUT / "human_groundedness_results.json", indent=2)
print(res)
print("\nDisagreements (> 1 point), resolved by discussion:")
print(m[m["diff"] > 1].to_string(index=False))